# Vibe Coding 演示：用自然语言 + Claude Code 写一段科研数据分析

**故事**：你想看看 arXiv 上「Large Language Model」相关论文最近 5 年的发文趋势。

Vibe Coding 模式：
1. 一句话把任务交给 AI
2. AI 写最小可跑版本
3. 你跑、你看输出、你记问题
4. 高频动作封装成 Skill

这个 Notebook 可以**不依赖 AI 服务**直接跑：代码已经写好，你重点看「Vibe Coding 应该长什么样」。

## 步骤 1：一句话定义任务

> 抓取 arXiv 上标题或摘要含 "large language model" 的论文，按月统计发文量，画出 2020–2025 的趋势图。要求：使用公开 API、不需要 key、画 PNG 保存到本目录。

下面是 AI 协作得到的最小可跑版本（讲师 / 学员可按需修改）。

In [ ]:
import urllib.request, urllib.parse, time
import xml.etree.ElementTree as ET
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt

NS = {"a": "http://www.w3.org/2005/Atom"}

def query_arxiv(keyword: str, max_results: int = 200):
    """调 arXiv API，返回 [(arxiv_id, title, published_date)]。"""
    params = {
        "search_query": f'all:"{keyword}"',
        "start": 0,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    url = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=30) as r:
        root = ET.fromstring(r.read())
    items = []
    for entry in root.findall("a:entry", NS):
        items.append((
            entry.findtext("a:id", default="", namespaces=NS),
            entry.findtext("a:title", default="", namespaces=NS).strip(),
            entry.findtext("a:published", default="", namespaces=NS),
        ))
    return items

papers = query_arxiv("large language model", max_results=200)
print(f"抓到 {len(papers)} 篇")
for p in papers[:3]:
    print("  -", p[2][:10], "|", p[1][:60])

## 步骤 2：调试记录

**第一次跑可能踩坑**：
- arXiv 限速：超过 1 次 / 3 秒会被限制
- `max_results` 单次最大 ≈ 2000，要多抓得分批

记下来 → 见 [调试记录模板.md](./调试记录模板.md)

## 步骤 3：按月统计 + 出图

In [ ]:
months = Counter()
for _, _, pub in papers:
    if not pub:
        continue
    dt = datetime.fromisoformat(pub.replace("Z", "+00:00"))
    months[(dt.year, dt.month)] += 1

keys = sorted(months.keys())
labels = [f"{y}-{m:02d}" for y, m in keys]
values = [months[k] for k in keys]

plt.figure(figsize=(10, 4))
plt.bar(labels, values)
plt.xticks(rotation=60, fontsize=8)
plt.title(f"arXiv 'large language model' submissions ({labels[0]} ~ {labels[-1]})")
plt.xlabel("month")
plt.ylabel("papers in sample")
plt.tight_layout()
plt.savefig("趋势.png", dpi=120)
plt.show()
print("已保存 趋势.png")

## 步骤 4：把高频代码动作封装成 Skill

上面的 `query_arxiv` 是高频动作：以后你想抓任何关键词、任何数量，都用得上。
把它写成 `~/.claude/skills/查arxiv/SKILL.md`，下次直接在 Claude Code 里说

> 用「查 arxiv」抓 50 篇 quantum computing 的论文

Claude Code 就会调用这个 Skill。

→ 详细做法见 [第 06 章 Skill 封装](../06-Skill封装/)。

## 验收

- [x] 一个可运行脚本
- [x] 调试记录
- [x] 结果图表
- [ ] 常用编程 Skill → 见第 06 章